In [2]:
from __future__ import annotations

import operator
from typing import Any, List, Optional, Union, Annotated
from typing import TypedDict

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph , START , END
from langgraph.types import Send

from langchain_openai import ChatOpenAI
from openai import OpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

import os
from langchain_google_genai import ChatGoogleGenerativeAI

from dotenv import load_dotenv
load_dotenv()


True

In [3]:
class Task(BaseModel):
    id : int
    title : str
    brief : str = Field(... , description="A brief description of the task")

In [4]:
class Plan(BaseModel):
    blog_title : str
    tasks : List[Task]

In [5]:
class State(TypedDict):
    topic : str
    plan : Plan

    sections = Annotated[List[str], operator.add]
    final : str

In [9]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.7  # 0.7 provides a good balance of creativity and structure
)

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    max_tokens=5  # Strictly limits the response length to save tokens
)

# 3. Send a single-word test prompt
print("Testing API connection...")
try:
    response = llm.invoke("Ping")
    
    # 4. Print the response and token usage breakdown
    print("\n Success! Connection established.")
    print(f"Model Response: {response.content}")
    print(f"Tokens Used: {response.usage_metadata}")

except Exception as e:
    print("\n Connection Failed!")
    print(f"Error Details: {e}")

Testing API connection...

 Success! Connection established.
Model Response: [{'type': 'text', 'text': 'Pong', 'extras': {'signature': 'EjQKMgEMOdbHkAIVO1eCGDkVUsWFvsYg/voGk+es7nlr+BHwFzvX/CW5kweK79Qsq4/qWXoH'}}]
Tokens Used: {'input_tokens': 2, 'output_tokens': 1, 'total_tokens': 3, 'input_token_details': {'cache_read': 0}}


In [ ]:
def orchestrator(state : State) -> dict:
    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(
                content = (
                    "Create a detailed blog plan with 7-10 sections on the following topic."
                )
            ),

            HumanMessage(
                content = f"Topic : {state['topic']}"
            )
        ]
    
    return plan
    )

In [ ]:
def fanout(state : State):
    return [Send("worker" , {"task" : task, "topic" :state["topic"] , plan : state["plan"]})
            for task in state["plan"].tasks]

In [ ]:
def worker(payload : dict) -> dict:

    #payload contain what we sent
    task = payload['task']
    topic = payload['topic']
    plan = payload['plan']

    blog_title = plan.blot_title

    section_md = llm.invoke(
        [
            SystemMessage(content = "Write one clean Markdown Section."),
            HumanMessage(
                content = (
                    f"Blog : {blog_title}\n"
                    f"Topic : {topic}\n\n"
                    f"Section : {task.title}\n"
                    f"Brief : {task.brief}\n\n"
                    "Return only the section content in the Markdown."
                )
            ),
        ]
    ).content.strip()

    return {"sections" : [section_md]}

In [ ]:
from pathlib import Path

def reducer(state : State) -> dict:

    title = state["plan"].blog_title
    body = "\n\n".join(state["sections"]).strip()

    final_md = f"# {title}\n\n {body}\n"

    filename = title.lower().replace(" ", "_")+ ".md"

    output_path = Path(filename)

    output_path.write_text(final_md , encoding = "utf-8")

    return {
        "final" : final_md
    }
